In [1]:
from pathlib import Path
import mne
import pandas as pd
from mne.preprocessing import fix_stim_artifact, ICA
from mne_faster import find_bad_channels, find_bad_epochs

## mne_tesa functionality
from mne_tesa import (tesa_interp_cubic_spline, tesa_replace_constant_amplitude, ICA_TESA, tesa_class_comp, 
                      tesa_ica_select, tesa_bandpass_filter_epochs, tesa_notch_filter_epochs
)
filename = 'example_data/TMS_EEG64Magventure80.vhdr' 

### An example of how to read in the TMS-EEG data

Below is an example of some different annotations for the TMS-pulse which some aquisition systems can use. 

If no annotations are available and no trigger was used you can also try this function:

```python
events_from_annot, event_dict = tesa_find_pulse(raw, sfreq, thresh=5, plot=False)

## And use the events in 
events, event_dict = mne.events_from_annotations(raw, event_id={desc: 1}
```

In [ ]:
raw = mne.io.read_raw_brainvision(filename, preload=True)


montage = mne.channels.make_standard_montage('standard_1020')
raw.set_montage(montage)

# S
event_descriptions = ["Response/R128", "Stimulus/A", "128", "191"]
events, event_dict = None, None

for desc in event_descriptions:
    try:
        events, event_dict = mne.events_from_annotations(raw, event_id={desc: 1})
        print(f"Successfully found events using description: '{desc}'")
        break
    except Exception:
        print(f"Could not find events for description: '{desc}'. Trying next...")


## Define interpolation bounds arond TMS-pulse
tmin_cut, tmax_cut = -0.002, 0.015

Extracting parameters from example_data/TMS_EEG64Magventure80.vhdr...
Setting channel info structure...
Reading 0 ... 1234099  =      0.000 ...   246.820 secs...
Used Annotations descriptions: [np.str_('Response/R128')]
Successfully found events using description: 'Response/R128'


## Epoched with a window of -1000 ms to 1000 ms around the TMS pulse, and the mean of each channel’s baseline (defined as a window of -500 to -10ms)

In [3]:
epochs = mne.Epochs(raw,
    events=events,
    tmin=-1,
    tmax=1,
    detrend=None,
    baseline=(-0.5, -0.01),
    preload=True,
)

Not setting metadata
113 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 113 events and 10001 original time points ...
0 bad epochs dropped


## TMS pulse artefact (the data from -2 to 15ms) was then removed and a cubic interpolation was applied to replace the missing data.

In [4]:
epochs = tesa_replace_constant_amplitude(
    epochs, inst_type="epochs", tmin=tmin_cut, tmax=tmax_cut, events=events
)


epochs = tesa_interp_cubic_spline(
    epochs,
    inst_type="epochs",
    tmin=tmin_cut,
    tmax=tmax_cut,
    interp_win=[0.001, 0.001],
    events=events,
)


Interpolated data between -0.002 and 0.015 using np.polyfit


### The recordings are down sampled to 1000 Hz

The polyphase method is used since this was the best way to replicate the down sampling in Matlab (EEGLAB)

In [5]:
epochs.resample(1000, method="polyphase")  #Hz

Polyphase resampling neighborhood: ±2 input samples


<Epochs | 113 events (all good), -1 – 0.999 s (baseline -0.5 – -0.01 s), ~110.4 MiB, data loaded,
 '1': 113>

## Following which, the trials and channels affected by prominent artefacts were detected by visual inspections and removed
### MNE-FASTER instead

In [6]:
epochs.info["bads"] = find_bad_channels(epochs, thres=3)
bad_epochs = find_bad_epochs(epochs, thres=3)
epochs.drop(bad_epochs)

Bad channel detection on EEG channels:
	Bad by variance: ['CP1', 'FT9']
	Bad by correlation: ['PO8']
	Bad by hurst: []
	Bad by kurtosis: ['Pz', 'FT9']
	Bad by line_noise: ['PO8']
Bad epoch detection on EEG channels:
	Bad by amplitude: [16]
	Bad by deviation: [10]
	Bad by variance: [11 16 35 37 46]
Dropped 6 epochs: 10, 11, 16, 35, 37, 46


<Epochs | 107 events (all good), -1 – 0.999 s (baseline -0.5 – -0.01 s), ~104.6 MiB, data loaded,
 '1': 107>

## Interpolated data around TMS pulse is replaced with constant amplitude data

In [7]:
epochs = tesa_replace_constant_amplitude(epochs, inst_type="epochs", tmin=tmin_cut, tmax=tmax_cut, events=events)

## TMS induced muscle and decay artefacts are identified using the FastICA algorithm and the ICA_TESA class

The parameters that control the TMS muscle and decay artefact detection are:

```python
tmsMuscle=True,
tmsMuscleThresh=8,
tmsMuscleWin=[0.011, 0.030],


```

In [ ]:
ica1 = ICA_TESA(n_components=30, noise_cov=None, 
           random_state=42, method='fastica', 
           fit_params=None, max_iter='auto', 
           allow_ref_meg=False, verbose=False)

ica1.fit(epochs)

suggested, comp_df = tesa_class_comp(
    epochs,
    ica1,
    tmsMuscle=True,
    tmsMuscleThresh=8,  # 8 was the default in TESA
    tmsMuscleWin=[0.011, 0.030],
    blink=False,
    blinkThresh=2.5,
    blinkElecs=["Fp1", "Fp2"],
    lat_eye_move=False,
    lat_eye_moveThresh=2.0,
    lat_eye_move_elecs=["F7", "F8"],
    persistant_muscle=False,
    persistant_muscle_thresh=0.5,  # original in TESA was -0.31,
    muscleFreqIn=[7, 70],  # original in TESA was [7, 70]
    # muscleFreqEx=[48, 52], ## no exclusion made in MNE implementation
    electrode_noise=False,
    elec_noise_thresh=4,epochs = tesa_interp_cubic_spline(
    epochs,
    inst_type="epochs",
    tmin=tmin_cut,
    tmax=tmax_cut,
    interp_win=[0.001, 0.001],
    events=events,
)
)

Fitting ICA to data using 60 channels (please be patient, this may take a while)


/tmp/ipykernel_217754/3540872091.py:6: RuntimeWarning: The epochs you passed to ICA.fit() were baseline-corrected. However, we suggest to fit ICA only on data that has been high-pass filtered, but NOT baseline-corrected.
  ica1.fit(epochs)


Selecting by number: 30 components
Fitting ICA took 5.0s.

 ****************** 
 Running component classification 
 ****************** 



### Ipywidgets in Jupyter notebook

The function below only works in jupyter notebooks. It is tested in jupyter lab and in the Jupyter extension for Visual Studio Code. 

It is possible to visualise the components, check if the thresholds selected above classified any of the components as artefacts (this can also be done by
inspecting the dataframe that tesa_class_comp returns). When you click "exclude" the component is added to the "exclude" list which is the return value 
of tesa_ica_select. This list can then be passed to the ICA.exclude list to update it with the excluded components. So the ICA will not be applied until 
explicitly called with ica.apply(epochs). 

When you exclude a component the widget will plot the overlay of the "cleaned" evoked object (black traces) over the the previous state of the evoked object (red traces).
When you exclude multiple components this overlay is updated to include multiple removals.  

In [ ]:
exclude = tesa_ica_select(epochs, ica1, suggested)

Dropdown(description='Component index:', options=('0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11'…

Button(description='Exclude', icon='check', style=ButtonStyle(), tooltip='Exclude ica component and plot overl…

Output()

## In this case no components were flagged as TMS-muscle artefact/decay artefact

In [ ]:
## This will display a dataframe view of suggested component
comp_df

In [ ]:
print(exclude)

In [ ]:
ica1.exclude = exclude
ica1.apply(epochs)

### Linear interpolation to replace the missing data
This was done in the original original paper by Biabani et al., 2024. Cubic spline interpolation could also be used here. 

This is mostly to demonstrate that the function already exists in MNE-Python

In [11]:
fix_stim_artifact(epochs, events=events, event_id=None, tmin=tmin_cut, tmax=tmax_cut, baseline=None, mode='linear', stim_channel=None, picks=None)

<Epochs | 107 events (all good), -1 – 0.999 s (baseline -0.5 – -0.01 s), ~104.6 MiB, data loaded,
 '1': 107>

### Band-pass (1-100 Hz) and band-stop (48-52 Hz) filtered using a zero-phase Butterworth filter (order = 2)

These filtering functions are pretty simple and were just developed to replicate the filtering that is used in Matlab

In [12]:
epochs = tesa_bandpass_filter_epochs(epochs, l_freq=1, h_freq=100, order=2)
epochs = tesa_notch_filter_epochs(epochs)

Constructing a 2 order bandpass filter with lfreq 1 and hfreq 100
Filtering data..
Constructing a 2 order bandstop filter with stopband (48, 52)
Filtering data..


### Other artefacts such as blink, lateral eye movement and persistant muscle artefacts are corrected by applying a second run of FastICA algorithm. Artefactual components are selected using the mne-tesa automated functions with default settings and checked visually

In [13]:
ica2 = ICA_TESA(n_components=30, noise_cov=None, 
           random_state=42, method='fastica', 
           fit_params=None, max_iter='auto', 
           allow_ref_meg=False, verbose=False)
ica2.fit(epochs)

suggested, comp_df2 = tesa_class_comp(
    epochs,
    ica2,
    tmsMuscle=True,
    tmsMuscleThresh=8,  # 8 was the default in TESA
    tmsMuscleWin=[0.011, 0.030],
    blink=True,
    blinkThresh=2.5,
    blinkElecs=["Fp1", "Fp2"],
    lat_eye_move=True,
    lat_eye_moveThresh=2.0,
    lat_eye_move_elecs=["F7", "F8"],
    persistant_muscle=True,
    persistant_muscle_thresh=0.5,  # original in TESA was -0.31,
    muscleFreqIn=[7, 70],  # original in TESA was [7, 70]
    # muscleFreqEx=[48, 52], ## no exclusion made in MNE implementation
    electrode_noise=True,
    elec_noise_thresh=4,
)

Fitting ICA to data using 60 channels (please be patient, this may take a while)


/tmp/ipykernel_217754/1508139590.py:5: RuntimeWarning: The epochs you passed to ICA.fit() were baseline-corrected. However, we suggest to fit ICA only on data that has been high-pass filtered, but NOT baseline-corrected.
  ica2.fit(epochs)


Selecting by number: 30 components
Fitting ICA took 5.1s.

 ****************** 
 Running component classification 
 ****************** 

Lateral eye movement detected in component 10
    Using multitaper spectrum estimation with 7 DPSS windows
Using EOG channel: Fp1
Using EOG channel: Fp2


In [16]:
comp_df2

,number,category
0,2,[blink]
1,3,[elecNoise]
2,5,"[persistant_muscle, elecNoise]"
3,6,"[persistant_muscle, elecNoise]"
4,7,[blink]
5,8,"[persistant_muscle, elecNoise]"
6,10,[lat_eye_move]
7,11,[elecNoise]
8,12,[elecNoise]
9,13,"[persistant_muscle, elecNoise]"


### As seen above one component was flagged as "blink", one as  "lateral eye movement" and several as "persistant muscle"

See below for example

In [14]:
exclude2 = tesa_ica_select(epochs, ica2, suggested)

Dropdown(description='Component index:', options=('0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11'…

Button(description='Exclude', icon='check', style=ButtonStyle(), tooltip='Exclude ica component and plot overl…

Output()

### Let's check how the evoked object would look if we remove some components

In [15]:
exclude2 = tesa_ica_select(epochs, ica2, suggested)

Dropdown(description='Component index:', options=('0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11'…

Button(description='Exclude', icon='check', style=ButtonStyle(), tooltip='Exclude ica component and plot overl…

Output()

In [21]:
print(f"Manually selected comopoents to exclude are:\n \n {exclude2}")

Manually selected comopoents to exclude are:
 
 [10, 2, 5, 6, 8, 13, 14, 21, 24, 26]


### Here we apply the ICA and remove the selected components from the epochs object

In [22]:
ica2.exclude = exclude2
ica2.apply(epochs)

Applying ICA to Epochs instance
    Transforming to ICA space (30 components)
    Zeroing out 10 ICA components
    Projecting back using 60 PCA components


/tmp/ipykernel_217754/3605377933.py:2: RuntimeWarning: The data you passed to ICA.apply() was baseline-corrected. Please note that ICA can introduce DC shifts, therefore you may wish to consider baseline-correcting the cleaned data again.
  ica2.apply(epochs)


<Epochs | 107 events (all good), -1 – 0.999 s (baseline -0.5 – -0.01 s), ~104.6 MiB, data loaded,
 '1': 107>

### Final interpolation of the artefact

In [23]:
epochs = tesa_interp_cubic_spline(
    epochs,
    inst_type="epochs",
    tmin=tmin_cut,
    tmax=tmax_cut,
    interp_win=[0.001, 0.001],
    events=events,
)

Interpolated data between -0.002 and 0.015 using np.polyfit


## Finally, the rejected channels are spatially interpolated using spherical method and all data were re-referenced to the common average

In [24]:
epochs.interpolate_bads()
epochs.set_eeg_reference('average')

Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 96.3 mm
Computing interpolation matrix from 60 sensor positions
Interpolating 4 sensors
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.


<Epochs | 107 events (all good), -1 – 0.999 s (baseline -0.5 – -0.01 s), ~104.6 MiB, data loaded,
 '1': 107>

## Plot interactive GFP

In [25]:
from plotly_evoked import tesa_plotly_gfp, tesa_plotly_evoked
tesa_plotly_gfp(epochs.average(), epochs.average().times, xlim=(-0.05, 0.3))

## Plot interactive Evoked plot

In [26]:
tesa_plotly_evoked(epochs.average(), epochs.times)

In [ ]:
# Save epochs
save_name = Path(filename).stem + '_preproc-epo.fif'
epochs.save(f"/output/path/{save_name}", overwrite=True)